In [1]:
import os
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"
import sys

PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

print("Using project root:", PROJECT_ROOT)

import torch
import albumentations as A
from albumentations.pytorch import ToTensorV2
from torch.utils.data import DataLoader
import torchmetrics
import numpy as np

import os
from mask_unet.models import get_segmentation_model, DebrisClassifier
from mask_unet.inference import inference_classifier_segmentation
from mask_unet.torchdataset import SegmentationDataset

Using project root: u:\users\beng\Mask_Unet


ModuleNotFoundError: No module named 'albumentations'

In [ ]:
def get_transforms():
    return A.Compose([
        A.Resize(384, 512),
        A.HorizontalFlip(p=0.5),
        A.RandomBrightnessContrast(p=0.2),
        A.Normalize(mean=(0.5,), std=(0.5,)),
        ToTensorV2()
    ])

In [ ]:
test_ds = SegmentationDataset(path_name='debris_data/test', transforms=get_transforms())
test_dataloader = DataLoader(test_ds, batch_size=4, shuffle=True)

In [ ]:
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

In [ ]:
model = get_segmentation_model()
model.to(DEVICE)

In [ ]:
checkpoint_name = 'Unet_debris_segmentation.pth'
model.load_state_dict(torch.load('models/' +checkpoint_name))

In [ ]:
pixel_accuracies = []
intersection_over_unions = []
metric_iou = torchmetrics.JaccardIndex(task='binary').to(DEVICE)

In [ ]:
with torch.no_grad():
    for inputs, targets in test_dataloader:
        inputs = inputs.to(DEVICE).float()
        targets = targets.to(DEVICE).float()

        outputs = model(inputs)
        predicted = (torch.sigmoid(outputs) > 0.5).float()

        correct = (predicted == targets).sum().item()
        total = torch.numel(targets)
        pixel_accuracies.append(correct / total)

        iou = metric_iou(predicted, targets.int())
        intersection_over_unions.append(iou.item())
pixel_accuracy = np.median(pixel_accuracies) * 100
iou_scores = np.median(intersection_over_unions) * 100
print(f"Median Pixel Accuracy: {np.median(pixel_accuracies) * 100:.2f}%")
print(f"Median IoU: {np.median(intersection_over_unions) * 100:.2f}%")

In [ ]:
with torch.no_grad():
    for i, (image_test, mask) in enumerate(test_dataloader):
        image_test = image_test.float().to(DEVICE)
        output = model(image_test)
        predicted_mask = (torch.sigmoid(output) > 0.5).float()

        fig, axs = plt.subplots(1, 3, figsize=(10, 10))
        axs[0].imshow(image_test[0][0].cpu(), cmap='gray')
        axs[1].imshow(mask[0][0].cpu(), cmap='gray')
        axs[2].imshow(predicted_mask[0][0].cpu(), cmap='gray')
        axs[0].set_title("image")
        axs[1].set_title("True Mask")
        axs[2].set_title("Predicted Mask")
